# DE 2d

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import numpy as np
import random
from invert_two_utils import (
    initialize_two_output_inversion,
    fitness_two_outputs,
    select_top_k_unique,
    unpack_two_output_solutions
)


def invert_two_outputs_de_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model1_path: str,
    model2_path: str,
    x_fixed_raw: np.ndarray,         # raw fixed inputs (len 10)
    actual_x_adj_raw: np.ndarray,    # raw adjustable inputs (len 4)
    target1_raw: float,
    target2_raw: float,
    pop_size: int,
    num_generations: int,
    F: float,                        # DE mutation factor
    CR: float,                       # DE crossover probability
    hidden_dims1: list[int],
    hidden_dims2: list[int],
    activation: str,
    x_adj_intervals_raw: np.ndarray, # raw [min,max] for adjustable inputs (4×2)
    Q_bounds: tuple[float, float],    # raw (Q_min, Q_max)
    k: int                            # number of best solutions to return
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    1. Load scalers, models, scale inputs, compute “actual” result,
       and build search bounds via initialize_two_output_inversion.
    2. Initialize a DE population of size pop_size in [min_vec, max_vec].
    3. Run num_generations of DE (mutation+cross+selection).
    4. Pick top-k unique scaled candidates with select_top_k_unique.
    5. Inverse‐scale candidates into raw inputs/predictions/residuals.
    """

    # [1] Shared initialization (scalers, models, actual, bounds, targets)
    init = initialize_two_output_inversion(
        x_scaler_path, y_scaler_path,
        model1_path, model2_path,
        hidden_dims1, hidden_dims2,
        activation,
        x_fixed_raw, actual_x_adj_raw,
        target1_raw, target2_raw,
        x_adj_intervals_raw, Q_bounds
    )
    model1, model2 = init["model1"], init["model2"]
    means_x, scales_x = init["means_x"], init["scales_x"]
    mean_y, scales_y = init["mean_y"], init["scales_y"]
    x_fixed_s = init["x_fixed_s"]
    results = init["results"]       # starts with actual test‐data result
    min_vec, max_vec = init["min_vec"], init["max_vec"]
    target_s = init["target_s"]     # scaled [target1, target2]

    # [2] DE population initialization
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]
    fitnesses  = [
        fitness_two_outputs(c, model1, model2, x_fixed_s, means_x, scales_x, target_s)
        for c in population
    ]

    # [3] Run Differential Evolution
    dim = len(min_vec)  # should be 4
    for _ in range(num_generations):
        for i in range(pop_size):
            # pick three distinct others
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c = random.sample(idxs, 3)
            x_a, x_b, x_c = population[a], population[b], population[c]

            # mutation: v = x_a + F*(x_b - x_c)
            v = x_a + F * (x_b - x_c)
            v = np.clip(v, min_vec, max_vec)

            # crossover → trial vector
            jrand = random.randrange(dim)
            trial = np.array([
                v[j] if (random.random() < CR or j == jrand) else population[i][j]
                for j in range(dim)
            ])
            trial = np.clip(trial, min_vec, max_vec)

            # selection
            f_trial = fitness_two_outputs(trial, model1, model2, x_fixed_s, means_x, scales_x, target_s)
            if f_trial < fitnesses[i]:
                population[i] = trial
                fitnesses[i]  = f_trial

    # [4] Select top-k unique scaled candidates
    best_scaled = select_top_k_unique(fitnesses, population, k)

    # [5] Inverse-scale to raw inputs, predictions, and residuals
    solved = unpack_two_output_solutions(
        best_scaled,
        model1, model2,
        means_x, scales_x,
        x_fixed_s,
        scales_y, mean_y,
        target1_raw, target2_raw
    )

    # Return the actual test‐data point + DE solutions
    return results + solved
